In [9]:
# Cell 1: Import Required Libraries
import pandas as pd
import numpy as np
import os
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
from typing import Dict, List, Tuple, Optional
import json
from pathlib import Path

# Cell 2: Building ID Generator Class (FIXED, Kigali = K)
class RwandaBuildingIDGenerator:
    """
    Generate unique building IDs for Rwanda national building dataset
    Format: K-G-KIM-BLDG-S1932923-E3010017
    """

    def __init__(self):
        # Rwanda coordinate bounds
        self.lat_bounds = (-3.0, -1.0)  # South bounds
        self.lon_bounds = (28.5, 31.0)  # East bounds

        # Initialize code mappings
        self.province_codes = {}
        self.district_codes = {}
        self.sector_codes = {}
        self.code_conflicts = {}

    def generate_location_codes(self, df: pd.DataFrame) -> Dict[str, Dict[str, str]]:
        """
        Generate unique codes for all administrative divisions in dataset
        """
        # Get unique administrative divisions
        provinces = df['province'].dropna().unique() if 'province' in df.columns else []
        districts = df['district'].dropna().unique() if 'district' in df.columns else []
        sectors = df['sector'].dropna().unique() if 'sector' in df.columns else []

        # Generate province codes (1 letter)
        for province in provinces:
            clean_name = self._clean_name(province)
            if clean_name:
                # Special case: Kigali city
                if clean_name.lower() in ['cityofkigali', 'kigali', 'kigalicity', 'citykigali']:
                    code = 'K'
                else:
                    code = clean_name[0].upper()
                self.province_codes[province] = code

        # Generate district codes with conflict resolution
        district_by_province = {}
        if 'province' in df.columns and 'district' in df.columns:
            for _, row in df[['province', 'district']].drop_duplicates().iterrows():
                if pd.notna(row['province']) and pd.notna(row['district']):
                    province = row['province']
                    district = row['district']
                    if province not in district_by_province:
                        district_by_province[province] = []
                    district_by_province[province].append(district)

        # Assign district codes
        for province, districts in district_by_province.items():
            used_codes = set()
            for district in districts:
                code = self._generate_unique_code(district, used_codes, max_length=2)
                self.district_codes[district] = code
                used_codes.add(code)

        # Generate sector codes with conflict resolution
        sector_by_district = {}
        if 'district' in df.columns and 'sector' in df.columns:
            for _, row in df[['district', 'sector']].drop_duplicates().iterrows():
                if pd.notna(row['district']) and pd.notna(row['sector']):
                    district = row['district']
                    sector = row['sector']
                    if district not in sector_by_district:
                        sector_by_district[district] = []
                    sector_by_district[district].append(sector)

        # Assign sector codes
        for district, sectors in sector_by_district.items():
            used_codes = set()
            for sector in sectors:
                code = self._generate_unique_code(sector, used_codes, max_length=3)
                self.sector_codes[sector] = code
                used_codes.add(code)

        return {
            'provinces': self.province_codes,
            'districts': self.district_codes,
            'sectors': self.sector_codes
        }

    def _clean_name(self, name: str) -> str:
        """Clean location name by removing special characters"""
        if pd.isna(name):
            return ""
        return re.sub(r'[^A-Za-z0-9]', '', str(name))

    def _generate_unique_code(self, name: str, used_codes: set, max_length: int = 3) -> str:
        """Generate unique code avoiding conflicts"""
        clean_name = self._clean_name(name).upper()
        if not clean_name:
            return "UNK"

        # Try single character first
        code = clean_name[0]
        if code not in used_codes:
            return code

        # Try two characters
        if len(clean_name) >= 2:
            code = clean_name[:2]
            if code not in used_codes:
                return code

        # Try three characters
        if len(clean_name) >= 3 and max_length >= 3:
            code = clean_name[:3]
            if code not in used_codes:
                return code

        # If still conflicts, add number
        for i in range(1, 10):
            test_code = f"{clean_name[0]}{i}"
            if test_code not in used_codes:
                return test_code

        return clean_name[:max_length]

    def process_coordinates(self, lat: float, lon: float) -> Tuple[str, str]:
        """
        Process coordinates to standard format - FIXED to ensure exactly 7 digits
        Returns: (lat_code, lon_code) e.g., ('S1932923', 'E3010017')
        """
        # Validate coordinates
        if pd.isna(lat) or pd.isna(lon):
            return ("S0000000", "E0000000")

        # Check if coordinates are within Rwanda bounds (removed warning prints for cleaner output)

        # Determine direction prefixes
        lat_prefix = 'S' if lat < 0 else 'N'
        lon_prefix = 'W' if lon < 0 else 'E'

        # Convert to 7-digit format - FIXED: ensure exactly 7 digits
        lat_value = int(abs(lat) * 10**7)
        lon_value = int(abs(lon) * 10**7)

        # Truncate to exactly 7 digits if longer
        lat_str = str(lat_value)[:7].ljust(7, '0')  # Pad with zeros if shorter
        lon_str = str(lon_value)[:7].ljust(7, '0')  # Pad with zeros if shorter

        lat_code = f"{lat_prefix}{lat_str}"
        lon_code = f"{lon_prefix}{lon_str}"

        return lat_code, lon_code

    def generate_building_id(self, row: pd.Series) -> str:
        """Generate building ID for a single row"""
        # Get location codes
        province_code = self.province_codes.get(row.get('province', ''), 'U')
        district_code = self.district_codes.get(row.get('district', ''), 'U')
        sector_code = self.sector_codes.get(row.get('sector', ''), 'UNK')

        # Process coordinates
        lat = row.get('latitude', row.get('lat', 0))
        lon = row.get('longitude', row.get('lon', row.get('lng', 0)))
        lat_code, lon_code = self.process_coordinates(lat, lon)

        # Build ID
        building_id = f"{province_code}-{district_code}-{sector_code}-BLDG-{lat_code}-{lon_code}"

        return building_id

    def decode_building_id(self, building_id: str) -> Dict[str, any]:
        """Decode building ID back to components"""
        parts = building_id.split('-')

        if len(parts) != 6:
            return {"error": "Invalid building ID format"}

        # Extract coordinate values
        lat_code = parts[4]
        lon_code = parts[5]

        # Decode latitude
        lat_sign = -1 if lat_code[0] == 'S' else 1
        lat_value = int(lat_code[1:]) / 10**7
        latitude = lat_sign * lat_value

        # Decode longitude
        lon_sign = -1 if lon_code[0] == 'W' else 1
        lon_value = int(lon_code[1:]) / 10**7
        longitude = lon_sign * lon_value

        return {
            "province_code": parts[0],
            "district_code": parts[1],
            "sector_code": parts[2],
            "latitude": latitude,
            "longitude": longitude,
            "lat_code": lat_code,
            "lon_code": lon_code
        }

# Cell 3: Data Processing Functions (unchanged)
def process_building_data(file_path: str, generator: RwandaBuildingIDGenerator) -> pd.DataFrame:
    """
    Process building data file and generate IDs
    """
    # Read file based on extension
    file_ext = Path(file_path).suffix.lower()

    if file_ext == '.csv':
        df = pd.read_csv(file_path)
    elif file_ext in ['.xlsx', '.xls']:
        df = pd.read_excel(file_path)
    else:
        raise ValueError(f"Unsupported file format: {file_ext}")

    print(f"Loaded {len(df)} buildings from {file_path}")
    print(f"Columns: {list(df.columns)}")

    # Standardize column names
    column_mapping = {
        'Province': 'province',
        'District': 'district',
        'Sector': 'sector',
        'Cell': 'cell',
        'Village': 'village',
        'Latitude': 'latitude',
        'Longitude': 'longitude',
        'lat': 'latitude',
        'lon': 'longitude',
        'lng': 'longitude'
    }

    df.rename(columns=column_mapping, inplace=True)

    # Generate location codes
    print("\nGenerating location codes...")
    codes = generator.generate_location_codes(df)

    # Display code mappings
    print("\nProvince Codes:")
    for name, code in codes['provinces'].items():
        print(f"  {name}: {code}")

    print("\nDistrict Codes:")
    for name, code in codes['districts'].items():
        print(f"  {name}: {code}")

    print(f"\nSector Codes (showing first 10):")
    for i, (name, code) in enumerate(codes['sectors'].items()):
        if i >= 10:
            print(f"  ... and {len(codes['sectors']) - 10} more")
            break
        print(f"  {name}: {code}")

    # Generate building IDs
    print("\nGenerating building IDs...")
    df['building_id'] = df.apply(generator.generate_building_id, axis=1)

    # Check for duplicates
    duplicates = df['building_id'].duplicated().sum()
    if duplicates > 0:
        print(f"\nWarning: {duplicates} duplicate IDs found!")
        duplicate_ids = df[df['building_id'].duplicated(keep=False)]['building_id'].unique()
        print(f"Duplicate IDs: {duplicate_ids[:5]}...")

    return df

# Cell 4: Validation and Statistics Functions (FIXED)
def validate_building_ids(df: pd.DataFrame) -> pd.DataFrame:
    """
    Validate generated building IDs and create report - FIXED validation regex
    """
    validation_results = []

    for idx, row in df.iterrows():
        building_id = row.get('building_id', '')
        issues = []

        # Check ID format
        parts = building_id.split('-')
        if len(parts) != 6:
            issues.append("Invalid format - wrong number of parts")
        else:
            # FIXED: Check coordinate format - exactly 1 letter + 7 digits
            if not re.match(r'^[NSEW]\d{7}$', parts[4]):
                issues.append("Invalid latitude format")
            if not re.match(r'^[NSEW]\d{7}$', parts[5]):
                issues.append("Invalid longitude format")

        # Check coordinates within Rwanda
        lat = row.get('latitude', 0)
        lon = row.get('longitude', 0)

        if pd.notna(lat) and not (-3.0 <= lat <= -1.0):
            issues.append(f"Latitude {lat} outside Rwanda bounds")
        if pd.notna(lon) and not (28.5 <= lon <= 31.0):
            issues.append(f"Longitude {lon} outside Rwanda bounds")

        validation_results.append({
            'index': idx,
            'building_id': building_id,
            'valid': len(issues) == 0,
            'issues': '; '.join(issues) if issues else 'None'
        })

    return pd.DataFrame(validation_results)

def convert_numpy_types(obj):
    """Convert numpy types to Python native types for JSON serialization"""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_numpy_types(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(item) for item in obj]
    else:
        return obj

def generate_statistics_report(df: pd.DataFrame) -> Dict:
    """
    Generate statistics report for the dataset
    """
    stats = {
        'total_buildings': int(len(df)),
        'unique_ids': int(df['building_id'].nunique()),
        'duplicates': int(df['building_id'].duplicated().sum()),
        'by_province': {},
        'by_district': {},
        'coordinate_stats': {}
    }

    # Convert value counts to regular Python dict with int values
    if 'province' in df.columns:
        province_counts = df['province'].value_counts()
        stats['by_province'] = {str(k): int(v) for k, v in province_counts.items()}

    if 'district' in df.columns:
        district_counts = df['district'].value_counts()
        stats['by_district'] = {str(k): int(v) for k, v in district_counts.items()}

    # Coordinate statistics
    if 'latitude' in df.columns:
        stats['coordinate_stats']['lat_min'] = float(df['latitude'].min())
        stats['coordinate_stats']['lat_max'] = float(df['latitude'].max())

    if 'longitude' in df.columns:
        stats['coordinate_stats']['lon_min'] = float(df['longitude'].min())
        stats['coordinate_stats']['lon_max'] = float(df['longitude'].max())

    return stats

# Cell 5: Main Processing Pipeline (Modified to return generator, save both xlsx and csv)
def main_pipeline(file_path: str, output_dir: str = "output"):
    """
    Main processing pipeline for building ID generation
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    # Initialize generator
    generator = RwandaBuildingIDGenerator()

    # Process data
    df = process_building_data(file_path, generator)

    # Generate validation report
    print("\nValidating building IDs...")
    validation_df = validate_building_ids(df)
    valid_count = len(validation_df[validation_df['valid']])
    invalid_count = len(validation_df[~validation_df['valid']])
    print(f"Valid buildings: {valid_count}")
    print(f"Invalid buildings: {invalid_count}")

    # Generate statistics
    stats = generate_statistics_report(df)
    print(f"\nDataset Statistics:")
    print(f"  Total buildings: {stats['total_buildings']}")
    print(f"  Unique IDs: {stats['unique_ids']}")
    print(f"  Duplicates: {stats['duplicates']}")

    # Save outputs
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Save processed data as CSV and XLSX
    output_file_csv = os.path.join(output_dir, f"buildings_with_ids_{timestamp}.csv")
    output_file_xlsx = os.path.join(output_dir, f"buildings_with_ids_{timestamp}.xlsx")
    df.to_csv(output_file_csv, index=False)
    df.to_excel(output_file_xlsx, index=False)
    print(f"\nSaved processed data to: {output_file_csv}")
    print(f"Saved processed data to: {output_file_xlsx}")

    # Save validation report as CSV and XLSX
    validation_file_csv = os.path.join(output_dir, f"validation_report_{timestamp}.csv")
    validation_file_xlsx = os.path.join(output_dir, f"validation_report_{timestamp}.xlsx")
    validation_df.to_csv(validation_file_csv, index=False)
    validation_df.to_excel(validation_file_xlsx, index=False)
    print(f"Saved validation report to: {validation_file_csv}")
    print(f"Saved validation report to: {validation_file_xlsx}")

    # Save code mappings
    code_mappings = {
        'provinces': generator.province_codes,
        'districts': generator.district_codes,
        'sectors': generator.sector_codes
    }
    mapping_file = os.path.join(output_dir, f"code_mappings_{timestamp}.json")
    with open(mapping_file, 'w') as f:
        json.dump(code_mappings, f, indent=2)
    print(f"Saved code mappings to: {mapping_file}")

    # Save statistics with numpy type conversion
    stats_converted = convert_numpy_types(stats)
    stats_file = os.path.join(output_dir, f"statistics_{timestamp}.json")
    with open(stats_file, 'w') as f:
        json.dump(stats_converted, f, indent=2)
    print(f"Saved statistics to: {stats_file}")

    return df, validation_df, stats, generator  # Return generator too

# Cell 6: Example Usage
# Process Gasabo data
gasabo_data_path = "gasabo-buildings-dev-subset.xlsx"

# Check if file exists
if os.path.exists(gasabo_data_path):
    # Load and inspect data
    gasabo_data = pd.read_excel(gasabo_data_path)
    print("Data loaded successfully!")
    print(f"\nColumns: {list(gasabo_data.columns)}")
    print(f"\nFirst row:")
    print(gasabo_data.iloc[0])

    # Process the data - now getting generator back
    df, validation_df, stats, generator = main_pipeline(gasabo_data_path)

    # Display sample results
    print("\nSample Building IDs Generated:")
    sample_df = df[['building_id', 'province', 'district', 'sector', 'latitude', 'longitude']].head(10)
    print(sample_df)

    # Show some validation results
    print("\nSample Validation Results:")
    print(validation_df[['building_id', 'valid', 'issues']].head(5))

else:
    print(f"File not found: {gasabo_data_path}")
    print("Creating sample data for demonstration...")

    # Create sample data
    sample_data = pd.DataFrame({
        'province': ['City of Kigali'] * 5,
        'district': ['Gasabo'] * 5,
        'sector': ['Kimironko', 'Kacyiru', 'Kimihurura', 'Remera', 'Gisozi'],
        'latitude': [-1.9329232, -1.9455, -1.9512, -1.9398, -1.9234],
        'longitude': [30.1001747, 30.0912, 30.0634, 30.1123, 30.0834],
        'cell': ['Cell1', 'Cell2', 'Cell3', 'Cell4', 'Cell5'],
        'village': ['Village1', 'Village2', 'Village3', 'Village4', 'Village5']
    })

    # Save sample data as CSV and XLSX
    sample_data.to_csv('sample_building_data.csv', index=False)
    sample_data.to_excel('sample_building_data.xlsx', index=False)
    print("Sample data saved to: sample_building_data.csv")
    print("Sample data saved to: sample_building_data.xlsx")

    # Process sample data
    df, validation_df, stats, generator = main_pipeline('sample_building_data.csv')

# Cell 7: Batch Processing for Multiple Districts
def process_multiple_districts(file_paths: List[str], output_dir: str = "output_all_districts"):
    """
    Process multiple district files and combine results
    """
    os.makedirs(output_dir, exist_ok=True)

    all_data = []
    generator = RwandaBuildingIDGenerator()  # Single generator for consistency

    for file_path in file_paths:
        if os.path.exists(file_path):
            print(f"\nProcessing: {file_path}")
            df = process_building_data(file_path, generator)
            all_data.append(df)
        else:
            print(f"Skipping: {file_path} (file not found)")

    # Combine all data
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"\nCombined dataset: {len(combined_df)} buildings")

        # Check for duplicate IDs across districts
        duplicates = combined_df['building_id'].duplicated().sum()
        if duplicates > 0:
            print(f"Warning: {duplicates} duplicate IDs found across districts!")

        # Save combined dataset as CSV and XLSX
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file_csv = os.path.join(output_dir, f"all_districts_with_ids_{timestamp}.csv")
        output_file_xlsx = os.path.join(output_dir, f"all_districts_with_ids_{timestamp}.xlsx")
        combined_df.to_csv(output_file_csv, index=False)
        combined_df.to_excel(output_file_xlsx, index=False)
        print(f"Saved combined data to: {output_file_csv}")
        print(f"Saved combined data to: {output_file_xlsx}")

        return combined_df

    return None

# Example usage for multiple districts
# district_files = [
#     "gasabo-buildings-dev-subset.xlsx",
#     "kicukiro-buildings.xlsx",
#     "nyarugenge-buildings.xlsx"
# ]
# combined_df = process_multiple_districts(district_files)

# Cell 8: ID Lookup and Verification Functions
def create_id_lookup_system(df: pd.DataFrame) -> Dict[str, pd.Series]:
    """
    Create a lookup system for quick ID verification and reverse lookup
    """
    lookup = {}

    for _, row in df.iterrows():
        building_id = row['building_id']
        lookup[building_id] = row

    return lookup

def find_buildings_in_area(df: pd.DataFrame, lat_center: float, lon_center: float,
                          radius_degrees: float = 0.001) -> pd.DataFrame:
    """
    Find all buildings within a certain radius of a coordinate
    """
    lat_min = lat_center - radius_degrees
    lat_max = lat_center + radius_degrees
    lon_min = lon_center - radius_degrees
    lon_max = lon_center + radius_degrees

    nearby = df[
        (df['latitude'] >= lat_min) &
        (df['latitude'] <= lat_max) &
        (df['longitude'] >= lon_min) &
        (df['longitude'] <= lon_max)
    ]

    return nearby

# Example usage
if 'df' in locals() and 'generator' in locals():
    # Create lookup system
    lookup = create_id_lookup_system(df)

    # Find buildings near a specific location
    nearby = find_buildings_in_area(df, -1.9329232, 30.1001747, radius_degrees=0.001)
    print(f"\nFound {len(nearby)} buildings in the area")

    # Show example of decoding a building ID
    if len(df) > 0:
        sample_id = df.iloc[0]['building_id']
        print(f"\nDecoding sample ID: {sample_id}")
        decoded = generator.decode_building_id(sample_id)
        print(f"Decoded components: {decoded}")

Data loaded successfully!

Columns: ['Unnamed: 0', 'Province', 'District', 'Sector', 'Cell', 'Village', 'latitude', 'longitude', 'geo']

First row:
Unnamed: 0                                                    4
Province                                         City of Kigali
District                                                 Gasabo
Sector                                                  Bumbogo
Cell                                                     Musave
Village                                                  Rebero
latitude                                              -1.916239
longitude                                             30.181252
geo           POLYGON ((30.1812732906637 -1.9162490171002, 3...
Name: 0, dtype: object
Loaded 305 buildings from gasabo-buildings-dev-subset.xlsx
Columns: ['Unnamed: 0', 'Province', 'District', 'Sector', 'Cell', 'Village', 'latitude', 'longitude', 'geo']

Generating location codes...

Province Codes:
  City of Kigali: K

District Codes:
 